In [0]:
from pyspark.sql import functions as F

orders = spark.table("samples.tpch.orders")   # no data is read yet

orders.printSchema()                           # column names and types

result = (
    orders
    .groupBy("o_orderstatus")
    .agg(
        F.count("*").alias("order_count"),
        F.sum("o_totalprice").alias("total_revenue"),
    )
    .orderBy(F.desc("total_revenue"))
)

result.show()                                  # data is read and computed here

In [0]:
# 4b. Translate your Day 1 Q2 (top 5 nations) into PySpark.
from pyspark.sql import functions as F

orders = spark.table("samples.tpch.orders")
customer = spark.table("samples.tpch.customer")
nation = spark.table("samples.tpch.nation")

result = (
    orders
    .join(customer, orders.o_custkey == customer.c_custkey)
    .join(nation, customer.c_nationkey == nation.n_nationkey)
    .groupBy(nation.n_name)
    .agg(F.sum(orders.o_totalprice).alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
    .limit(5)
)

display(result)
result.explain()

In [0]:
#4c (stretch). LAG in PySpark. Rebuild task 2a:

from pyspark.sql.window import Window
from pyspark.sql import functions as F

orders = spark.table("samples.tpch.orders")

w = Window.partitionBy("o_custkey").orderBy("o_orderdate", "o_orderkey")

orders_with_prev = orders.withColumn("prev_order_date", F.lag("o_orderdate").over(w))

orders_with_days = orders_with_prev.withColumn(
    "days_since_prev",
    F.datediff(F.col("o_orderdate"), F.col("prev_order_date"))
)

display(orders_with_days.limit(10))
# days_since_prev is now a column in orders_with_days